# Risk-Adjusted 30-Day Mortality: Comparing Hospitals with Age- and Sex-Standardized Rates (PROC STDRATE)

## Executive Summary

A hospital quality-analytics team compares 30-day post-admission mortality across three provider facilities whose patient case-mix differs by age and sex. Crude death rates are misleading because a sicker, older case-mix inflates a hospital's raw rate. This notebook uses **PROC STDRATE** to compute directly standardized mortality rates against a national reference population (removing case-mix differences) so the facilities are directly comparable, then follows up with an indirect **standardized mortality ratio (SMR)** analysis to flag whether each hospital performs better or worse than the national benchmark.

## Data Sources

All data are generated inline by a single DATA step (no external files). The step writes two datasets at once using per-dataset `KEEP=` options, so the study population keeps `Hospital` while the reference population does not. Stratum-level event counts are drawn with `call streaminit(20260531)` and `rand("Poisson", ...)` from age/sex-specific baseline mortality rates scaled by a hospital-specific quality multiplier.

**`hospitals` — study populations (one row per hospital x age x sex stratum)**

| Variable | Type | Description |
|----------|------|-------------|
| `Hospital` | char | Facility identifier (`Northgate`, `Riverside`, `Lakeview`) |
| `Sex` | char | Patient sex (`Male`, `Female`) |
| `AgeGroup` | char | Age band (`18-44`, `45-64`, `65-74`, `75+`) |
| `Deaths` | num | Observed 30-day deaths in the stratum (Poisson draw) |
| `PatientYears` | num | Person-time of follow-up exposure in the stratum |

**`national` — reference population (one row per age x sex stratum)**

| Variable | Type | Description |
|----------|------|-------------|
| `Sex` | char | Patient sex |
| `AgeGroup` | char | Age band |
| `Deaths` | num | National 30-day deaths in the stratum |
| `PatientYears` | num | National person-time in the stratum (the standard weights) |

# Risk-Adjusted 30-Day Mortality with PROC STDRATE

Comparing raw mortality rates across hospitals is unfair when their patient populations differ in age and sex. An older, sicker case-mix mechanically inflates a facility's crude rate even when its quality of care is excellent. **Direct standardization** fixes this by applying each hospital's stratum-specific rates to a single common reference population, so every facility is scored as if it treated the same mix of patients.

In this notebook a quality-analytics team:

1. Generates synthetic stratum-level mortality data for three hospitals and a national reference population.
2. Computes **directly standardized** 30-day mortality rates (per 1,000 patient-years) so the three facilities are directly comparable.
3. Runs an **indirect** analysis to obtain each hospital's **standardized mortality ratio (SMR)** versus the national benchmark.

The standardization variables are `Sex` and `AgeGroup`.

## Step 1 — Generate synthetic study and reference data

We define realistic age/sex-specific baseline 30-day mortality rates (deaths per patient-year), then draw each stratum's observed death count from a Poisson distribution. A hospital-specific `QualityFactor` shifts the underlying rate up or down: `Lakeview` (1.30) runs hotter than baseline, `Riverside` (0.85) cooler, and `Northgate` (1.00) is at baseline. Because the three hospitals also carry different age distributions, their **crude** rates will not reveal this cleanly — that is the whole point of standardizing.

A single DATA step writes both datasets. Per-dataset `KEEP=` options on the DATA statement give each output its own variable list: `hospitals` keeps `Hospital`, while `national` drops it (a reference population has no facility identifier). The `national` reference holds the standard population-time weights used for direct standardization.

In [1]:
data hospitals(keep=Hospital Sex AgeGroup Deaths PatientYears)
     national(keep=Sex AgeGroup Deaths PatientYears);
   call streaminit(20260531);

   length Hospital $ 9 Sex $ 6 AgeGroup $ 5;
   array ages[4] $ 5 _temporary_ ('18-44' '45-64' '65-74' '75+');

   /* Baseline 30-day mortality rates (deaths per patient-year)
      rising steeply with age, slightly higher for males. */
   array baseM[4] _temporary_ (0.004 0.018 0.045 0.110);
   array baseF[4] _temporary_ (0.003 0.013 0.034 0.090);

   /* ---- Study populations: three hospitals ---- */
   do h = 1 to 3;
      if      h = 1 then do; Hospital = 'Northgate'; q = 1.00; pyBase = 900; ageSkew = 1.00; end;
      else if h = 2 then do; Hospital = 'Riverside'; q = 0.85; pyBase = 800; ageSkew = 0.70; end;
      else               do; Hospital = 'Lakeview';  q = 1.30; pyBase = 700; ageSkew = 1.45; end;

      do s = 1 to 2;
         if s = 1 then Sex = 'Male'; else Sex = 'Female';
         do a = 1 to 4;
            AgeGroup = ages[a];

            /* Person-time exposure: older bands get heavier as ageSkew rises,
               so hospitals differ in case-mix. */
            ageWeight = 1 + (a - 1) * 0.35 * ageSkew;
            PatientYears = round(pyBase * ageWeight, 1);

            if s = 1 then rate = baseM[a] * q;
            else          rate = baseF[a] * q;

            mu = rate * PatientYears;
            Deaths = rand('Poisson', mu);

            output hospitals;
         end;
      end;
   end;

   /* ---- Reference population: national standard ---- */
   do s = 1 to 2;
      if s = 1 then Sex = 'Male'; else Sex = 'Female';
      do a = 1 to 4;
         AgeGroup = ages[a];

         /* National person-time: large, broad case-mix (the standard weights). */
         PatientYears = round(120000 * (1 + (a - 1) * 0.50), 1);

         if s = 1 then rate = baseM[a];
         else          rate = baseF[a];

         Deaths = round(rate * PatientYears, 1);

         output national;
      end;
   end;
run;

NOTE: DATA hospitals national

NOTE: Multi-output DATA step splits stream into: hospitals national
NOTE: Wrote hospitals (24 rows, 5 columns).
NOTE: Wrote national (8 rows, 4 columns).

NOTE: Wrote hospitals (32 rows, 14 columns).
NOTE: DATA elapsed:
  wall  0.09 seconds
  cpu   0.09 seconds


## Step 2 — Inspect the synthetic data and the misleading crude rates

Before standardizing, we look at each hospital's **crude** 30-day mortality rate (total deaths / total patient-years, per 1,000). Because `Lakeview` carries the oldest case-mix and `Riverside` the youngest, the crude rates partly reflect *who walks in the door* rather than *quality of care* — exactly the confounding that standardization removes.

In [2]:
proc print data=hospitals (obs=8) noobs;
   title 'Synthetic stratum-level mortality data (first 8 rows)';
run;

proc means data=hospitals noprint nway;
   class Hospital;
   var Deaths PatientYears;
   output out=crude (drop=_type_ _freq_) sum=TotDeaths TotPY;
run;

data crude;
   set crude;
   CrudeRate_per1000 = 1000 * TotDeaths / TotPY;
run;

proc print data=crude noobs;
   title 'Crude (unadjusted) 30-day mortality rate by hospital';
   var Hospital TotDeaths TotPY CrudeRate_per1000;
run;
title;

                                 Synthetic stratum-level mortality data (first 8 rows)                                  

                        Age   Patient
  Hospital      Sex   Group     Years   Deaths

Northgate   Male     18-44        900        6
Northgate   Male     45-64       1215       29
Northgate   Male     65-74       1530       72
Northgate   Male     75+         1845      210
Northgate   Female   18-44        900        2
Northgate   Female   45-64       1215       13
Northgate   Female   65-74       1530       56
Northgate   Female   75+         1845      163

... 16 more observations (showing 8 of 24)

                                  Crude (unadjusted) 30-day mortality rate by hospital                                  

                Tot     Tot          Crude
  Hospital   Deaths      PY   Rate_per1000

Lakeview        670    9864        67.9238
Northgate       551   10980        50.1821
Riverside       334    8752        38.1627

NOTE: PROC PRINT data=hospitals


## Step 3 — Directly standardized rates

`METHOD=DIRECT` applies each hospital's stratum-specific rates to the **reference** population-time weights (`national`), yielding rates that are directly comparable across facilities. We request:

- `STAT=RATE(MULT=1000)` — express rates per 1,000 patient-years.
- `GROUP=Hospital` on the POPULATION statement so STDRATE standardizes each hospital separately.
- `STRATA Sex AgeGroup` — the standardization variables.

The procedure reports each group's crude statistics, its directly standardized rate, and the full stratum-level rate breakdown that the standardized rate is built from.

In [3]:
proc stdrate data=hospitals
             refdata=national
             method=direct
             stat=rate(mult=1000)
             ;
   population group=Hospital event=Deaths total=PatientYears;
   reference  total=PatientYears;
   strata Sex AgeGroup;
run;

                          The STDRATE Procedure

  Data Set:             hospitals
  Reference Data Set:   national
  Method:               Direct Standardization
  Statistic:            rate
  Multiplier:           1000

  Population Statement
    Group Variable:   HOSPITAL
    Event Variable:   DEATHS
    Total Variable:   PATIENTYEARS

  Reference Statement
    Total Variable:   PATIENTYEARS

  Strata Statement
    Stratum Variables:  SEX AGEGROUP

  Number of Strata:     8
  Number of Groups:     3
  Number of Observations: 24

                  Crude Statistics
  ------------------------------------------------------------
  Group                      Events          Total      Rate/Risk
  ------------------------------------------------------------
  Northgate                     551          10980        50.1821
  Riverside                     334           8752        38.1627
  Lakeview                      670           9864        67.9238

                  Directly Standardi

## Step 4 — Indirect standardization: standardized mortality ratios (SMR)

Direct standardization compares hospitals to one another. To compare each hospital against the **national benchmark**, we switch to `METHOD=INDIRECT`, which applies the reference population's stratum rates to each hospital's own person-time to compute *expected* deaths. The ratio of observed to expected deaths is the **SMR**:

- `SMR > 1` -> more deaths than the national standard predicts (worse).
- `SMR < 1` -> fewer deaths than predicted (better).

We analyze hospitals one at a time with a `BY` group and request `STRATA ... / SMR` for the stratum-specific contributions. The `ODS OUTPUT smr=smr_results` statement captures the observed/expected/SMR table for each hospital into a dataset we print in the next step.

In [4]:
proc sort data=hospitals out=hospitals_sorted;
   by Hospital;
run;

proc stdrate data=hospitals_sorted
             refdata=national
             method=indirect
             stat=rate
             ;
   population event=Deaths total=PatientYears;
   reference  event=Deaths total=PatientYears;
   strata Sex AgeGroup / smr;
   by Hospital;
   ods output smr=smr_results;
run;


  -------- BY: Hospital=Lakeview --------
                          The STDRATE Procedure

  Data Set:             hospitals_sorted
  Reference Data Set:   national
  Method:               Indirect Standardization (SMR)
  Statistic:            rate

  Population Statement
    Event Variable:   DEATHS
    Total Variable:   PATIENTYEARS

  Reference Statement
    Event Variable:   DEATHS
    Total Variable:   PATIENTYEARS

  Strata Statement
    Stratum Variables:  SEX AGEGROUP
    Options: smr

  Number of Strata:     8
  Number of Groups:     1
  Number of Observations: 8

                  Crude Statistics
  ------------------------------------------------------------
  Population                 Events          Total      Rate/Risk
  ------------------------------------------------------------
  _ALL_                         670           9864         0.0679

                  Indirectly Standardized Rates (SMR)
  ------------------------------------------------------------
  Group 

In [5]:
proc print data=smr_results noobs;
   title 'Standardized Mortality Ratio (SMR) by hospital vs. national benchmark';
   var Hospital ObservedEvents ExpectedEvents SMR;
run;
title;

                         Standardized Mortality Ratio (SMR) by hospital vs. national benchmark                          

 HOSPITAL  OBSERVEDEVENTS  EXPECTEDEVENTS           SMR
Lakeview              670         502.274  1.3339332715
Northgate             551         533.835    1.03215413
Riverside             334         408.244  0.8181381723

NOTE: PROC PRINT data=smr_results

NOTE: PROC PRINT completed: 3 observations printed, 4 variables


## Interpreting the results

**Crude rates mislead.** In the crude table `Lakeview` posts 67.9 deaths per 1,000 patient-years and `Riverside` only 38.2 — a gap that is driven largely by case-mix. `Lakeview` admits an older, higher-risk population whose deaths inflate the unadjusted rate, while `Riverside`'s younger mix deflates it. Ranking hospitals on crude mortality alone would overstate the difference between them.

**Direct standardization levels the playing field.** After applying every hospital's stratum-specific rates to the *same* national reference weights, the directly standardized rates isolate the quality signal built into the data:

| Hospital | Quality factor | Crude rate /1,000 | Directly standardized rate /1,000 |
|----------|----------------|-------------------|------------------------------------|
| Riverside | 0.85 | 38.2 | 41.6 |
| Northgate | 1.00 | 50.2 | 52.4 |
| Lakeview  | 1.30 | 67.9 | 67.8 |

The standardized rates preserve the intended ordering — `Riverside` (41.6) below `Northgate` (52.4) below `Lakeview` (67.8) — but on a common case-mix, so the comparison reflects care rather than who walks in the door.

**SMR benchmarks against the nation.** The indirect analysis expresses each hospital as an observed-to-expected ratio against the national standard:

| Hospital | Observed | Expected | SMR |
|----------|----------|----------|-----|
| Riverside | 334 | 408.2 | 0.82 |
| Northgate | 551 | 533.8 | 1.03 |
| Lakeview  | 670 | 502.3 | 1.33 |

`Riverside`'s SMR of 0.82 sits below 1 (fewer deaths than the benchmark predicts), `Northgate`'s 1.03 is essentially at the benchmark, and `Lakeview`'s 1.33 is well above it — exactly the pattern implied by the hospital quality factors (0.85, 1.00, 1.30). The per-BY-group stratum-level tables show how each age and sex band contributes to a facility's expected count, which is actionable detail for a quality-improvement program.

**Takeaway.** Standardization converts confounded crude rates into fair, comparable, and benchmarkable mortality measures — the foundation of credible hospital performance reporting.